# Module 2 · Lesson 03: Chain-of-Thought Prompting

**Chain-of-Thought (CoT)** prompting asks the model to "think step by step".
This dramatically improves accuracy on reasoning, math, and logic tasks.

## What you will learn
1. Why CoT works (and when it doesn't)
2. Zero-shot CoT ("think step by step")
3. Few-shot CoT (reasoning examples)
4. CoT for **code debugging**
5. CoT for **decision making**
6. Different **trigger phrases**

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
client = OpenAI()
 
def ask(prompt, system=None, temperature=0, max_tokens=15000):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

---
## 1. The Problem: Without CoT

Let's see what happens when we ask a tricky math problem *without* CoT:

In [2]:
prompt = """A store has 15 apples. It sells 7 in the morning and receives a delivery of 12.
Then it sells 5 more in the afternoon. How many apples remain?
"""
direct_answer = ask(f"{prompt}\n\nAnswer with just the number")
print(f"Direct answer (no CoT): {direct_answer}")



Direct answer (no CoT): 15


---
## 2. Zero-Shot CoT — "Let's think step by step"

Simply adding **"Let's think step by step"** triggers reasoning:

In [3]:
cot_answer = ask(f"{prompt}\n\nLet's think step by step. Use plain text, not Latex")
display(Markdown(f"### Step-by-step Reasoning\n\n{cot_answer.strip()}"))

### Step-by-step Reasoning

Sure! Let's break it down step by step:

1. The store starts with 15 apples.
2. In the morning, it sells 7 apples. 
   - So, 15 - 7 = 8 apples remain.
3. Then, the store receives a delivery of 12 apples.
   - Now, 8 + 12 = 20 apples.
4. In the afternoon, it sells 5 more apples.
   - So, 20 - 5 = 15 apples remain.

Therefore, the store has 15 apples left.

In [4]:
problem = """John has 3 boxes.
Each box contains 5 bags.
Each bag contains 4 marbles.
 
He gives away 18 marbles to his friend.
 
Then he buys 2 more boxes with the same structure.
 
How many marbles dooes he have now?
"""

In [5]:
direct_ans = ask(f"{problem}\n\n Answer with just the number.", temperature=0)
print(direct_ans)

62


In [6]:
cot_ans = ask(f"{problem}\n\nLet's think step by step. Use plain text, not Latex", temperature=0)
# print(cot_ans)
display(Markdown(cot_ans))

Let's break it down step by step:

1. **Calculate the total number of marbles John initially has:**
   - John has 3 boxes.
   - Each box contains 5 bags.
   - Each bag contains 4 marbles.

   First, calculate the total number of bags:
   Total bags = Number of boxes × Bags per box = 3 × 5 = 15 bags.

   Now, calculate the total number of marbles:
   Total marbles = Total bags × Marbles per bag = 15 × 4 = 60 marbles.

2. **John gives away 18 marbles:**
   - Marbles after giving away = Initial marbles - Marbles given away
   - Marbles after giving away = 60 - 18 = 42 marbles.

3. **John buys 2 more boxes:**
   - Each new box also contains 5 bags, and each bag contains 4 marbles.
   - Total new boxes = 2.
   - Total new bags = New boxes × Bags per box = 2 × 5 = 10 bags.
   - Total new marbles = New bags × Marbles per bag = 10 × 4 = 40 marbles.

4. **Calculate the total number of marbles now:**
   - Total marbles now = Marbles after giving away + New marbles
   - Total marbles now = 42 + 40 = 82 marbles.

So, John now has 82 marbles.

> 💡 **The research:** Kojima et al. (2022) showed that adding "Let's think step by step" 
> improved accuracy by 10–40% on reasoning benchmarks — with zero examples!

---
## 3. Logical Reasoning

In [7]:
logic = """Consider these statements:
1. All programmers drink coffee.
2. Some coffee drinkers are night owls.
3. No night owls wake up early.
 
Question: Can we coclude thet some programmers don't wake up early?
 
Think thhought this step by step, examining each logic connection.
"""
 
result = ask(logic)
display(Markdown(result))

Let's analyze the statements step by step to determine if we can conclude that some programmers don't wake up early.

1. **All programmers drink coffee.**
   - This means that every individual who is a programmer is also a coffee drinker. We can represent this as:
     - If \( P \) is a programmer, then \( P \) drinks coffee.

2. **Some coffee drinkers are night owls.**
   - This indicates that there is at least one individual who drinks coffee and is also a night owl. We can represent this as:
     - There exists at least one individual \( C \) such that \( C \) drinks coffee and \( C \) is a night owl.

3. **No night owls wake up early.**
   - This means that if someone is a night owl, then that individual does not wake up early. We can represent this as:
     - If \( N \) is a night owl, then \( N \) does not wake up early.

Now, let's connect these statements:

- From statement 1, we know that all programmers drink coffee. Therefore, all programmers are included in the group of coffee drinkers.
- From statement 2, we know that some coffee drinkers are night owls. This means that there exists at least one coffee drinker who is also a night owl.
- Since all programmers are coffee drinkers, it is possible that some of these coffee drinkers (who are also night owls) could be programmers.

However, we cannot definitively conclude that "some programmers don't wake up early" based solely on the information provided. Here's why:

- While it is true that some coffee drinkers are night owls (and thus do not wake up early), we do not know if those specific coffee drinkers who are night owls are programmers. 
- It is possible that all night owls who drink coffee are not programmers, in which case all programmers could potentially wake up early.

Thus, we cannot conclude that some programmers don't wake up early based on the given statements. The conclusion is not necessarily true; it is possible but not guaranteed.

---
## 4. CoT for Code Debugging

CoT is particularly effective for finding bugs — it forces the model to trace execution:

In [8]:
debug_prompt = """Find the bug in this Python code:
 
```python
def find_duplicates(lst):
    seen = set()
    duplicates = set()
    for item in lst:
        if item in seen:
            duplicates.add(item)
    seen.add(item)
    return list(duplicates)
 
# Expected: [2, 3]
# Actual:   []
print(find_duplicates([1, 2, 2, 3, 3, 4]))
```
 
Trace through the code step by step with the input [1, 2, 2, 3, 3, 4]."""
 
result = ask(debug_prompt)
display(Markdown(result))

Let's trace through the provided code step by step with the input list `[1, 2, 2, 3, 3, 4]`.

Here is the code again for reference:

```python
def find_duplicates(lst):
    seen = set()
    duplicates = set()
    for item in lst:
        if item in seen:
            duplicates.add(item)
        seen.add(item)
    return list(duplicates)

# Expected: [2, 3]
# Actual:   []
print(find_duplicates([1, 2, 2, 3, 3, 4]))
```

### Step-by-Step Execution:

1. **Initialization**:
   - `seen = set()`
   - `duplicates = set()`

2. **First Iteration** (`item = 1`):
   - Check if `1` is in `seen`: `1 not in seen` (False)
   - Add `1` to `seen`: `seen = {1}`
   - `duplicates` remains `{}`.

3. **Second Iteration** (`item = 2`):
   - Check if `2` is in `seen`: `2 not in seen` (False)
   - Add `2` to `seen`: `seen = {1, 2}`
   - `duplicates` remains `{}`.

4. **Third Iteration** (`item = 2`):
   - Check if `2` is in `seen`: `2 in seen` (True)
   - Add `2` to `duplicates`: `duplicates = {2}`
   - Add `2` to `seen`: `seen = {1, 2}` (no change since `2` is already there).

5. **Fourth Iteration** (`item = 3`):
   - Check if `3` is in `seen`: `3 not in seen` (False)
   - Add `3` to `seen`: `seen = {1, 2, 3}`
   - `duplicates` remains `{2}`.

6. **Fifth Iteration** (`item = 3`):
   - Check if `3` is in `seen`: `3 in seen` (True)
   - Add `3` to `duplicates`: `duplicates = {2, 3}`
   - Add `3` to `seen`: `seen = {1, 2, 3}` (no change since `3` is already there).

7. **Sixth Iteration** (`item = 4`):
   - Check if `4` is in `seen`: `4 not in seen` (False)
   - Add `4` to `seen`: `seen = {1, 2, 3, 4}`
   - `duplicates` remains `{2, 3}`.

### Conclusion:
After the loop completes, the function returns `list(duplicates)`, which is `list({2, 3})`. However, the code has a bug in the indentation of the line `seen.add(item)`. It should be inside the loop, but it is currently outside. This means that the last item (`4`) is added to `seen` after the loop has finished, which is not the intended behavior.

### Fix:
To fix the bug, the line `seen.add(item)` should be indented to be part of the loop:

```python
def find_duplicates(lst):
    seen = set()
    duplicates = set()
    for item in lst:
        if item in seen:
            duplicates.add(item)
        seen.add(item)  # This line should be inside the loop
    return list(duplicates)
```

With this change, the function will correctly identify duplicates and return `[2, 3]` as expected.

---
## 5. Few-Shot CoT

Providing **reasoning examples** teaches the model HOW to think, not just what to output:

In [9]:
prompt = """Solve word problems step by step. Use plain text, not LaTeX.
 
Q: If John has 3 boxes with 5 apples each, and he gives away 4 apples, how many does he have?
A: Step 1: John starts with 3 x 5 = 15 apples.
   Step 2: He gives away 4, so 15 - 4 = 11 apples.
   Answer: 11
 
Q: A train travels 60 km/h for 2 hours, then 90 km/h for 1 hour. What's the total distance?
A: Step 1: First leg: 60 x 2 = 120 km
   Step 2: Second leg: 90 x 1 = 90 km
   Step 3: Total: 120 + 90 = 210 km.
   Answer: 210 km
 
Q: A rectangle has a perimeter of 30 cm. If the length is twice the width, what is the area?
A:"""
 
result = ask(prompt)
display(Markdown(f"### Few-Shot CoT Result\n\n{result}"))

### Few-Shot CoT Result

Step 1: Let the width be W cm. Then the length is 2W cm.

Step 2: The formula for the perimeter of a rectangle is P = 2(length + width). So, we have:
30 = 2(2W + W)

Step 3: Simplifying the equation: 
30 = 2(3W) 
30 = 6W

Step 4: Solving for W: 
W = 30 / 6 = 5 cm

Step 5: Now, find the length: 
Length = 2W = 2 x 5 = 10 cm

Step 6: Calculate the area using the formula Area = length x width: 
Area = 10 x 5 = 50 cm²

Answer: 50 cm²

---
## 6. CoT for Decision Making

In [10]:
decision = """We need to choose a database for our new project.
 
Requirements:
- Flexible schema (data structure may change)
- High write throughtput (IoT sensor data)
- Horizontal scaling needed
- Team has SQL experience
- Budet is limited
 
Options: PostgreSQL, MongoDB, DynamoDB, TimescaleDB
 
Analyse each option against our requirements step by step, then recommend one."""
 
result = ask(decision, max_tokens=1500)
display(Markdown(result))

Let's analyze each of the database options against your requirements step by step:

### 1. PostgreSQL
- **Flexible Schema**: PostgreSQL supports JSONB, which allows for some flexibility in schema design. However, it is primarily a relational database, so it may not be as flexible as NoSQL options.
- **High Write Throughput**: PostgreSQL can handle a decent amount of write operations, but it may not be optimized for extremely high write throughput compared to some NoSQL databases.
- **Horizontal Scaling**: PostgreSQL traditionally has limitations in horizontal scaling, although there are extensions like Citus that can help with this.
- **Team SQL Experience**: Your team will likely be comfortable with PostgreSQL due to their SQL experience.
- **Budget**: PostgreSQL is open-source and free to use, which is beneficial for a limited budget.

### 2. MongoDB
- **Flexible Schema**: MongoDB is a NoSQL database that allows for a flexible schema, making it easy to adapt to changing data structures.
- **High Write Throughput**: MongoDB is designed for high write throughput and can handle large volumes of data, making it suitable for IoT sensor data.
- **Horizontal Scaling**: MongoDB supports sharding, which allows for horizontal scaling across multiple servers.
- **Team SQL Experience**: While MongoDB uses a query language that is different from SQL, the team may need to learn some new concepts, but it is generally considered user-friendly.
- **Budget**: MongoDB has a free tier (Community Edition), but costs can increase with the use of their managed services (Atlas).

### 3. DynamoDB
- **Flexible Schema**: DynamoDB is a NoSQL database that offers a flexible schema, allowing for easy changes to data structures.
- **High Write Throughput**: DynamoDB is designed for high write throughput and can handle massive amounts of data, making it ideal for IoT applications.
- **Horizontal Scaling**: DynamoDB automatically scales horizontally, which is a significant advantage for growing applications.
- **Team SQL Experience**: DynamoDB uses a different query language (partially SQL-like with PartiQL), so the team may need to adapt, but it is manageable.
- **Budget**: DynamoDB has a pay-as-you-go pricing model, which can become expensive depending on usage, especially for high write throughput.

### 4. TimescaleDB
- **Flexible Schema**: TimescaleDB is built on PostgreSQL and supports time-series data, allowing for some flexibility in schema design, especially for time-series data.
- **High Write Throughput**: TimescaleDB is optimized for time-series data and can handle high write throughput effectively.
- **Horizontal Scaling**: TimescaleDB supports horizontal scaling through its multi-node capabilities, but it may not be as straightforward as some NoSQL options.
- **Team SQL Experience**: Since TimescaleDB is based on PostgreSQL, your team will be comfortable with it.
- **Budget**: TimescaleDB is open-source and free to use, which is beneficial for a limited budget.

### Recommendation
Considering your requirements:

- **Flexible Schema**: MongoDB and DynamoDB excel here, but TimescaleDB also offers flexibility for time-series data.
- **High Write Throughput**: MongoDB and DynamoDB are strong contenders, with TimescaleDB also performing well for time-series data.
- **Horizontal Scaling**: DynamoDB is the best option for automatic horizontal scaling, but MongoDB and TimescaleDB also support it.
- **Team SQL Experience**: PostgreSQL and TimescaleDB are the best fits here, but MongoDB is relatively easy to learn.
- **Budget**: PostgreSQL and TimescaleDB are the most budget-friendly options.

### Final Recommendation
**TimescaleDB** is the recommended option for your project. It meets the requirements for flexible schema, high write throughput for time-series data, and your team's SQL experience. Additionally, it is budget-friendly as it is open-source. If your data is primarily time-series (as is common with IoT sensor data), TimescaleDB will provide the best performance and ease of use for your team.

---
## 7. CoT Trigger Phrases

Different phrases work for different scenarios:

| Trigger | Best For |
|---------|----------|
| "Let's think step by step" | General reasoning |
| "Walk me through your reasoning" | Complex analyses |
| "Show your work" | Math problems |
| "Trace through the code" | Debugging |
| "Consider each option" | Decision making |
| "First, identify the key facts" | Reading comprehension |

In [11]:
triggers = [
    "Let's think step by step.",
    "Show your work.",
    "Walk me through your reasoning.",
]
 
problem = "What is 15% of 240?"
 
for trigger in triggers:
    result = ask(f"{problem} {trigger}", max_tokens=1500)
    display(Markdown(f"### Trigger: \"{trigger}\""))
    # print(f"Answer:  {result[:1000]}...\n")
    display(Markdown(f"### Answer:\n\n{result}"))
    print("-" * 71)

### Trigger: "Let's think step by step."

### Answer:

To find 15% of 240, you can follow these steps:

1. **Convert the percentage to a decimal**: 
   - 15% can be written as a decimal by dividing by 100: 
   \[
   15\% = \frac{15}{100} = 0.15
   \]

2. **Multiply the decimal by the number**: 
   - Now, multiply 0.15 by 240:
   \[
   0.15 \times 240
   \]

3. **Perform the multiplication**:
   - You can do this calculation:
   \[
   0.15 \times 240 = 36
   \]

So, 15% of 240 is **36**.

-----------------------------------------------------------------------


### Trigger: "Show your work."

### Answer:

To find 15% of 240, you can follow these steps:

1. Convert the percentage to a decimal. To do this, divide 15 by 100:
   \[
   15\% = \frac{15}{100} = 0.15
   \]

2. Multiply the decimal by the number (240):
   \[
   0.15 \times 240
   \]

3. Perform the multiplication:
   \[
   0.15 \times 240 = 36
   \]

Therefore, 15% of 240 is **36**.

-----------------------------------------------------------------------


### Trigger: "Walk me through your reasoning."

### Answer:

To find 15% of 240, you can follow these steps:

1. **Convert the percentage to a decimal**: 
   - 15% can be written as a decimal by dividing by 100. 
   - So, 15% = 15 / 100 = 0.15.

2. **Multiply the decimal by the number**: 
   - Now, multiply 0.15 by 240.
   - Calculation: \( 0.15 \times 240 \).

3. **Perform the multiplication**:
   - \( 0.15 \times 240 = 36 \).

Therefore, 15% of 240 is **36**.

-----------------------------------------------------------------------


---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **Zero-shot CoT** | Add "think step by step" — free accuracy boost |
| **Few-shot CoT** | Show reasoning examples for complex tasks |
| **Best for** | Math, logic, debugging, multi-step problems |
| **Not needed for** | Simple factual questions, translation |
| **Trade-off** | More tokens used (longer output) = higher cost |

---
**Next:** `04_pctf_framework.ipynb` — A systematic framework for writing effective prompts